# Fare Amount Prediction

This notebook builds the second part of usecase2: predicting expected NYC taxi `fare_amount` before ride initiation.

Each code cell is now separated into a specific task, and each task has a markdown explanation for what it does, why it is needed, and how it works.

Production note: during training, historical `trip_duration` is used. In the final app, this value should come from your completed trip-duration prediction model.

# 1. Import Core Libraries

**What this cell does:** Imports basic libraries for paths, warnings, data handling, plotting, and model saving.

**Why we do this:** The notebook needs these tools before any data processing or modeling can happen.

**How it works:** `pathlib` handles paths, `pandas` and `numpy` handle data, `matplotlib` and `seaborn` create plots, and `joblib` saves the final pipeline.

In [ ]:
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

# 2. Import Scikit-Learn Tools

**What this cell does:** Imports preprocessing, model, feature selection, metric, split, and pipeline classes from scikit-learn.

**Why we do this:** These are the main building blocks for a clean machine learning workflow.

**How it works:** `ColumnTransformer` handles different column types, `Pipeline` connects preprocessing and models, `RFE` performs feature selection, and metrics evaluate results.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 3. Import Optional Boosting Models

**What this cell does:** Tries to import XGBoost and LightGBM if they are installed.

**Why we do this:** Your project already uses boosting models, and they often perform well on structured taxi data.

**How it works:** The notebook sets the model variable to `None` if the package is missing, so the rest of the notebook can still run.

In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

# 4. Set Project Paths

**What this cell does:** Finds the usecase2 project directory, data path, model directory, and saved model path.

**Why we do this:** Jupyter may be opened from different folders, so fixed relative paths can break.

**How it works:** The code checks common launch locations and then creates the `models` directory if needed.

In [ ]:
cwd = Path.cwd().resolve()

if (cwd / "data" / "cleaned_taxi_data.parquet").exists():
    PROJECT_DIR = cwd
elif (cwd / "USECASE2_Trip_Fare_Forecasting" / "data" / "cleaned_taxi_data.parquet").exists():
    PROJECT_DIR = cwd / "USECASE2_Trip_Fare_Forecasting"
else:
    PROJECT_DIR = cwd.parent

DATA_PATH = PROJECT_DIR / "data" / "cleaned_taxi_data.parquet"
MODEL_DIR = PROJECT_DIR / "models"
FARE_MODEL_PATH = MODEL_DIR / "fare_prediction_pipeline.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data path:", DATA_PATH)
print("Model output path:", FARE_MODEL_PATH)

# 5. Load Cleaned Dataset

**What this cell does:** Loads the cleaned taxi dataset from parquet format.

**Why we do this:** This file contains the `fare_amount` target, while the previous duration feature file does not.

**How it works:** `pd.read_parquet()` reads the data, then shape and first rows are displayed.

In [ ]:
df = pd.read_parquet(DATA_PATH)

print("Dataset Shape:", df.shape)
display(df.head())

# 6. Check Dataset Columns

**What this cell does:** Displays the column names and data types.

**Why we do this:** Before selecting features, we should confirm which columns exist and whether types look correct.

**How it works:** `info()` prints column names, non-null counts, and data types.

In [ ]:
display(df.info())

# 7. Define Target

**What this cell does:** Stores the fare column name in one variable.

**Why we do this:** Using one target variable makes later code easier to read and change.

**How it works:** The model will learn to predict this column from the input features.

In [ ]:
target = "fare_amount"

# 8. Summarize Fare Target

**What this cell does:** Shows descriptive statistics for fare amount.

**Why we do this:** This helps identify scale, skew, and possible outlier values before modeling.

**How it works:** `describe()` returns count, mean, standard deviation, min, quartiles, and max.

In [ ]:
display(df[target].describe())

# 9. Plot Fare Distribution

**What this cell does:** Plots the distribution of fare amount.

**Why we do this:** Fare values usually have a long tail, and the plot makes that visible.

**How it works:** A histogram shows frequency and a KDE curve shows the distribution shape.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[target], bins=80, kde=True)
plt.title("Fare Amount Distribution")
plt.xlabel("Fare Amount")
plt.ylabel("Trip Count")
plt.show()

# 10. Copy Data for Modeling

**What this cell does:** Creates a separate fare modeling DataFrame.

**Why we do this:** This protects the original dataset from accidental changes during feature engineering.

**How it works:** `df.copy()` creates an independent working copy called `fare_df`.

In [ ]:
fare_df = df.copy()

# 11. Convert Pickup Datetime

**What this cell does:** Converts pickup datetime into pandas datetime format.

**Why we do this:** Datetime features cannot be extracted reliably unless the column is a proper datetime type.

**How it works:** `pd.to_datetime()` converts the pickup column.

In [ ]:
fare_df["tpep_pickup_datetime"] = pd.to_datetime(fare_df["tpep_pickup_datetime"])

# 12. Create Basic Time Features

**What this cell does:** Creates pickup hour, day of week, and day of month.

**Why we do this:** Taxi fare behavior can depend on time because traffic and trip patterns change by hour and day.

**How it works:** Pandas datetime accessors extract these values from pickup time.

In [ ]:
fare_df["pickup_hour"] = fare_df["tpep_pickup_datetime"].dt.hour
fare_df["pickup_dayofweek"] = fare_df["tpep_pickup_datetime"].dt.dayofweek
fare_df["pickup_day"] = fare_df["tpep_pickup_datetime"].dt.day

# 13. Create Weekend and Rush-Hour Flags

**What this cell does:** Creates binary features for weekend and rush hour.

**Why we do this:** These simple flags help the model capture traffic-related fare behavior.

**How it works:** Weekend is based on day-of-week, and rush hour is based on common morning/evening busy hours.

In [ ]:
fare_df["is_weekend"] = fare_df["pickup_dayofweek"].isin([5, 6]).astype(int)
fare_df["is_rush_hour"] = fare_df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# 14. Create Location Pair

**What this cell does:** Creates a pickup-dropoff route feature.

**Why we do this:** A route pair gives the model more information than pickup and dropoff zones separately.

**How it works:** Pickup and dropoff IDs are joined into a string like `239_161`.

In [ ]:
fare_df["Location_Pair"] = (
    fare_df["PULocationID"].astype(str) + "_" + fare_df["DOLocationID"].astype(str)
)

# 15. Select Modeling Columns

**What this cell does:** Keeps only columns that will be used for fare prediction.

**Why we do this:** This removes unused columns and avoids leakage from information not known before or during ride setup.

**How it works:** The list includes trip details, engineered features, route features, and the target.

In [ ]:
modeling_columns = [
    "passenger_count", "trip_distance", "RatecodeID",
    "PULocationID", "DOLocationID", "trip_duration",
    "cbd_congestion_fee", "pickup_hour", "pickup_dayofweek",
    "pickup_day", "is_weekend", "is_rush_hour",
    "Location_Pair", "fare_amount",
]

fare_df = fare_df[modeling_columns]

# 16. Drop Missing Values

**What this cell does:** Removes rows with missing modeling values.

**Why we do this:** Most scikit-learn models cannot train directly with missing values.

**How it works:** `dropna()` removes records where any selected modeling column is missing.

In [ ]:
fare_df = fare_df.dropna()
print("After dropping missing values:", fare_df.shape)

# 17. Remove Invalid Trip Records

**What this cell does:** Filters out records with impossible fare, distance, or duration.

**Why we do this:** Negative or zero fare/distance/duration records can damage model learning.

**How it works:** Boolean filters keep only realistic positive values.

In [ ]:
fare_df = fare_df[
    (fare_df["fare_amount"] > 0)
    & (fare_df["trip_distance"] > 0)
    & (fare_df["trip_duration"] > 0)
]

print("Final modeling shape:", fare_df.shape)
display(fare_df.head())

# 18. Split X and y

**What this cell does:** Separates input features and target values.

**Why we do this:** Models need `X` as the input table and `y` as the value to predict.

**How it works:** `X` drops `fare_amount`; `y` keeps only `fare_amount`.

In [ ]:
X = fare_df.drop(columns=[target])
y = fare_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

# 19. Create Train and Temporary Split

**What this cell does:** Creates the first split for training and later validation/test data.

**Why we do this:** Training data fits the model, while non-training data checks generalization.

**How it works:** `train_test_split` keeps 70 percent for training and 30 percent as temporary holdout.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("Train:", X_train.shape)
print("Temporary holdout:", X_temp.shape)

# 20. Create Validation and Test Split

**What this cell does:** Splits temporary holdout data into validation and test sets.

**Why we do this:** Validation is used for model selection; test is used only for final evaluation.

**How it works:** The temporary holdout is split equally into validation and test data.

In [ ]:
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

# 21. Create Training Sample

**What this cell does:** Creates a smaller training subset for faster notebook runs.

**Why we do this:** The full dataset is large, and model comparison/RFE can be slow. Sampling keeps experiments practical.

**How it works:** If training rows exceed the sample size, a reproducible random sample is used.

In [ ]:
MODEL_TRAIN_SAMPLE_SIZE = 600_000

if MODEL_TRAIN_SAMPLE_SIZE is not None and len(X_train) > MODEL_TRAIN_SAMPLE_SIZE:
    train_sample_index = X_train.sample(MODEL_TRAIN_SAMPLE_SIZE, random_state=42).index
    X_fit = X_train.loc[train_sample_index]
    y_fit = y_train.loc[train_sample_index]
else:
    X_fit = X_train
    y_fit = y_train

print("Rows used for fitting:", X_fit.shape[0])

# 22. Define Numeric Features

**What this cell does:** Lists numeric input columns.

**Why we do this:** Numeric columns can be scaled or passed directly depending on the model type.

**How it works:** This list is reused in preprocessing, RFE, and final saving.

In [ ]:
numeric_features = [
    "passenger_count", "trip_distance", "RatecodeID",
    "trip_duration", "cbd_congestion_fee", "pickup_hour",
    "pickup_dayofweek", "pickup_day", "is_weekend", "is_rush_hour",
]

# 23. Define Categorical Features

**What this cell does:** Lists categorical input columns.

**Why we do this:** Location IDs and route pairs are categories, not continuous measurements.

**How it works:** These columns will be converted with one-hot encoding.

In [ ]:
categorical_features = ["PULocationID", "DOLocationID", "Location_Pair"]

# 24. Build One-Hot Helper

**What this cell does:** Creates a compatibility helper for dense one-hot encoding.

**Why we do this:** Different scikit-learn versions use different parameter names.

**How it works:** The function tries `sparse_output=False`, then falls back to `sparse=False`.

In [ ]:
def make_onehot_encoder(**kwargs):
    try:
        return OneHotEncoder(sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=False, **kwargs)

# 25. Build Linear Preprocessor

**What this cell does:** Creates preprocessing for linear models.

**Why we do this:** Linear models usually perform better when numeric features are scaled.

**How it works:** Numeric columns use `StandardScaler`; categorical columns use `OneHotEncoder`.

In [ ]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), categorical_features),
    ]
)

# 26. Build Tree Preprocessor

**What this cell does:** Creates preprocessing for tree-based models.

**Why we do this:** Tree models do not need scaled numeric values, but still need categorical encoding.

**How it works:** Numeric columns are passed through and categorical columns are one-hot encoded.

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), categorical_features),
    ]
)

# 27. Define Regression Metrics

**What this cell does:** Creates one function for MAE, RMSE, and R2.

**Why we do this:** Using one metric function keeps all model evaluations consistent.

**How it works:** The function compares actual values with predicted values and returns a dictionary.

In [ ]:
def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }

# 28. Define Baseline Linear Models

**What this cell does:** Creates Linear Regression and Ridge Regression pipelines.

**Why we do this:** Linear Regression gives a simple baseline; Ridge adds regularization to reduce overfitting.

**How it works:** Each pipeline combines preprocessing and model training.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", LinearRegression()),
    ]),
    "Ridge Regression": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", Ridge(alpha=10.0, random_state=42)),
    ]),
}

# 29. Add XGBoost Model

**What this cell does:** Adds a regularized XGBoost model if installed.

**Why we do this:** XGBoost often works well for tabular data but can overfit, so regularization is important.

**How it works:** Depth, child weight, row sampling, column sampling, alpha, lambda, and gamma control complexity.

In [ ]:
if XGBRegressor is not None:
    models["XGBoost"] = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=700, max_depth=4, learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.80, min_child_weight=20,
            reg_alpha=0.5, reg_lambda=5.0, gamma=0.2,
            objective="reg:squarederror", random_state=42, n_jobs=-1,
        )),
    ])

# 30. Add LightGBM Model

**What this cell does:** Adds a regularized LightGBM model if installed.

**Why we do this:** LightGBM is fast and accurate, but regularization helps avoid overfitting.

**How it works:** Leaf count, depth, child samples, row/column sampling, alpha, and lambda limit model complexity.

In [ ]:
if LGBMRegressor is not None:
    models["LightGBM"] = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=700, num_leaves=31, max_depth=8, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, min_child_samples=100,
            reg_alpha=0.5, reg_lambda=5.0, random_state=42, n_jobs=-1,
        )),
    ])

print("Models to train:", list(models.keys()))

# 31. Train Models

**What this cell does:** Fits all initial candidate models.

**Why we do this:** Training lets each model learn the relationship between ride features and fare amount.

**How it works:** Each pipeline handles preprocessing first, then model fitting.

In [ ]:
trained_models = {}

for model_name, pipeline in models.items():
    print(f"Training {model_name}...")
    pipeline.fit(X_fit, y_fit)
    trained_models[model_name] = pipeline

# 32. Compare Train and Validation Performance

**What this cell does:** Calculates train and validation metrics to detect overfitting.

**Why we do this:** A model is likely overfitting if train error is much lower than validation error.

**How it works:** The RMSE gap percentage is flagged when validation RMSE is more than 15 percent worse than training RMSE.

In [ ]:
results = []

for model_name, pipeline in trained_models.items():
    train_pred = pipeline.predict(X_fit)
    valid_pred = pipeline.predict(X_valid)

    train_metrics = regression_metrics(y_fit, train_pred)
    valid_metrics = regression_metrics(y_valid, valid_pred)

    metrics = {
        "Model": model_name,
        "Train_MAE": train_metrics["MAE"],
        "Valid_MAE": valid_metrics["MAE"],
        "Train_RMSE": train_metrics["RMSE"],
        "Valid_RMSE": valid_metrics["RMSE"],
        "Train_R2": train_metrics["R2"],
        "Valid_R2": valid_metrics["R2"],
    }
    metrics["RMSE_Gap"] = metrics["Valid_RMSE"] - metrics["Train_RMSE"]
    metrics["RMSE_Gap_%"] = (metrics["RMSE_Gap"] / metrics["Train_RMSE"]) * 100
    metrics["Overfit_Flag"] = metrics["RMSE_Gap_%"] > 15
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values(["Valid_RMSE", "RMSE_Gap_%"])
display(results_df)

# 33. Create RFE Sample

**What this cell does:** Creates a smaller sample for Recursive Feature Elimination.

**Why we do this:** RFE can be expensive on millions of rows and many encoded route columns.

**How it works:** A reproducible sample is used to make feature selection practical.

In [ ]:
RFE_SAMPLE_SIZE = 100_000

if len(X_fit) > RFE_SAMPLE_SIZE:
    rfe_sample_index = X_fit.sample(RFE_SAMPLE_SIZE, random_state=42).index
    X_rfe = X_fit.loc[rfe_sample_index]
    y_rfe = y_fit.loc[rfe_sample_index]
else:
    X_rfe = X_fit
    y_rfe = y_fit

print("RFE sample shape:", X_rfe.shape)

# 34. Build RFE Preprocessor

**What this cell does:** Prepares numeric data for RFE.

**Why we do this:** RFE requires numeric arrays, so categorical features must be encoded.

**How it works:** Numeric features are scaled and categorical features are one-hot encoded with capped categories.

In [ ]:
rfe_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", make_onehot_encoder(handle_unknown="ignore", max_categories=30), categorical_features),
])

X_rfe_encoded = rfe_preprocessor.fit_transform(X_rfe)
rfe_feature_names = rfe_preprocessor.get_feature_names_out()

print("Encoded RFE shape:", X_rfe_encoded.shape)

# 35. Run RFE

**What this cell does:** Runs Recursive Feature Elimination using Ridge Regression.

**Why we do this:** RFE helps remove weak/noisy features and can reduce overfitting.

**How it works:** RFE repeatedly trains a model, removes weaker features, and keeps the strongest 60 percent.

In [ ]:
n_features_to_select = max(10, int(len(rfe_feature_names) * 0.60))

rfe_selector = RFE(
    estimator=Ridge(alpha=10.0, random_state=42),
    n_features_to_select=n_features_to_select,
    step=0.10,
)

rfe_selector.fit(X_rfe_encoded, y_rfe)

print("Total transformed features:", len(rfe_feature_names))
print("Selected transformed features:", rfe_selector.support_.sum())

# 36. Display RFE Results

**What this cell does:** Shows selected transformed features and rankings.

**Why we do this:** This lets you inspect which encoded variables were kept by RFE.

**How it works:** Selected features have `selected=True`; smaller ranking means stronger feature.

In [ ]:
rfe_results = pd.DataFrame({
    "transformed_feature": rfe_feature_names,
    "selected": rfe_selector.support_,
    "ranking": rfe_selector.ranking_,
}).sort_values(["ranking", "transformed_feature"])

display(rfe_results.head(40))

# 37. Map RFE Features to Raw Columns

**What this cell does:** Defines a helper to map encoded feature names back to original columns.

**Why we do this:** The final model should use normal input columns, not internal one-hot names.

**How it works:** The function detects whether an encoded feature belongs to a categorical group or numeric feature.

In [ ]:
def get_raw_feature_name(transformed_feature):
    feature_name = transformed_feature.split("__", 1)[-1]

    for categorical_feature in categorical_features:
        if feature_name == categorical_feature or feature_name.startswith(f"{categorical_feature}_"):
            return categorical_feature

    return feature_name

# 38. Build Final Selected Feature List

**What this cell does:** Creates the final raw feature list selected after RFE.

**Why we do this:** This feature list will be used for final model training and production prediction.

**How it works:** Selected transformed RFE features are mapped back to raw numeric and categorical column names.

In [ ]:
selected_transformed_features = rfe_results.loc[
    rfe_results["selected"], "transformed_feature"
].tolist()

selected_raw_features = sorted(
    {get_raw_feature_name(feature) for feature in selected_transformed_features},
    key=lambda feature: numeric_features.index(feature)
    if feature in numeric_features
    else len(numeric_features) + categorical_features.index(feature),
)

selected_numeric_features = [f for f in numeric_features if f in selected_raw_features]
selected_categorical_features = [f for f in categorical_features if f in selected_raw_features]
final_selected_features = selected_numeric_features + selected_categorical_features

print("Selected raw features for final model:")
for feature in final_selected_features:
    print("-", feature)

# 39. Define Final Preprocessor Builder

**What this cell does:** Creates a reusable function for selected-feature preprocessors.

**Why we do this:** After RFE, selected features may be fewer than original features, so preprocessors must match them.

**How it works:** The function adds numeric and categorical transformers only when selected lists are not empty.

In [ ]:
def build_preprocessor(selected_numeric, selected_categorical, scale_numeric=False):
    transformers = []

    if selected_numeric:
        numeric_transformer = StandardScaler() if scale_numeric else "passthrough"
        transformers.append(("num", numeric_transformer, selected_numeric))

    if selected_categorical:
        transformers.append((
            "cat",
            OneHotEncoder(handle_unknown="ignore", min_frequency=50),
            selected_categorical,
        ))

    return ColumnTransformer(transformers=transformers)

# 40. Build Final Preprocessors

**What this cell does:** Creates final preprocessors using RFE-selected features.

**Why we do this:** Final models must use only the selected feature columns.

**How it works:** A scaled preprocessor is made for Ridge and an unscaled numeric preprocessor for tree models.

In [ ]:
selected_linear_preprocessor = build_preprocessor(
    selected_numeric_features, selected_categorical_features, scale_numeric=True
)

selected_tree_preprocessor = build_preprocessor(
    selected_numeric_features, selected_categorical_features, scale_numeric=False
)

# 41. Define Final Models

**What this cell does:** Creates final model candidates after RFE.

**Why we do this:** This checks which model performs best after feature reduction.

**How it works:** The same regularized model logic is used, but now with selected-feature preprocessors.

In [ ]:
final_models = {
    "RFE Ridge Regression": Pipeline([
        ("preprocessor", selected_linear_preprocessor),
        ("model", Ridge(alpha=10.0, random_state=42)),
    ])
}

if XGBRegressor is not None:
    final_models["RFE XGBoost"] = Pipeline([
        ("preprocessor", selected_tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=700, max_depth=4, learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.80, min_child_weight=20,
            reg_alpha=0.5, reg_lambda=5.0, gamma=0.2,
            objective="reg:squarederror", random_state=42, n_jobs=-1,
        )),
    ])

if LGBMRegressor is not None:
    final_models["RFE LightGBM"] = Pipeline([
        ("preprocessor", selected_tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=700, num_leaves=31, max_depth=8, learning_rate=0.05,
            subsample=0.85, colsample_bytree=0.85, min_child_samples=100,
            reg_alpha=0.5, reg_lambda=5.0, random_state=42, n_jobs=-1,
        )),
    ])

print("Final models to train:", list(final_models.keys()))

# 42. Train Final Models

**What this cell does:** Fits final models using only RFE-selected features.

**Why we do this:** This is the final training stage before selecting the production model.

**How it works:** Each final pipeline trains on `X_fit[final_selected_features]`.

In [ ]:
final_trained_models = {}

for model_name, pipeline in final_models.items():
    print(f"Training {model_name}...")
    pipeline.fit(X_fit[final_selected_features], y_fit)
    final_trained_models[model_name] = pipeline

# 43. Evaluate Final Models

**What this cell does:** Compares final RFE models and checks overfitting again.

**Why we do this:** A final model should have strong validation performance and a reasonable train-validation gap.

**How it works:** The same metrics and overfit flag are calculated for the RFE models.

In [ ]:
final_results = []

for model_name, pipeline in final_trained_models.items():
    train_pred = pipeline.predict(X_fit[final_selected_features])
    valid_pred = pipeline.predict(X_valid[final_selected_features])

    train_metrics = regression_metrics(y_fit, train_pred)
    valid_metrics = regression_metrics(y_valid, valid_pred)

    metrics = {
        "Model": model_name,
        "Train_MAE": train_metrics["MAE"],
        "Valid_MAE": valid_metrics["MAE"],
        "Train_RMSE": train_metrics["RMSE"],
        "Valid_RMSE": valid_metrics["RMSE"],
        "Train_R2": train_metrics["R2"],
        "Valid_R2": valid_metrics["R2"],
    }
    metrics["RMSE_Gap"] = metrics["Valid_RMSE"] - metrics["Train_RMSE"]
    metrics["RMSE_Gap_%"] = (metrics["RMSE_Gap"] / metrics["Train_RMSE"]) * 100
    metrics["Overfit_Flag"] = metrics["RMSE_Gap_%"] > 15
    final_results.append(metrics)

final_results_df = pd.DataFrame(final_results).sort_values(["Valid_RMSE", "RMSE_Gap_%"])
display(final_results_df)

# 44. Select Best Final Model

**What this cell does:** Selects the best model from final RFE results.

**Why we do this:** The final model should be selected from validation results, not training results.

**How it works:** The first row after sorting by validation RMSE is chosen as the best model.

In [ ]:
selection_results_df = final_results_df
selection_models = final_trained_models
prediction_features = final_selected_features

best_model_name = selection_results_df.iloc[0]["Model"]
best_model = selection_models[best_model_name]

print("Best Model:", best_model_name)
print("Features used:", prediction_features)

# 45. Test Best Model

**What this cell does:** Evaluates the selected model on untouched test data.

**Why we do this:** The test set gives the final unbiased estimate of performance.

**How it works:** The best model predicts `X_test`, then MAE, RMSE, and R2 are calculated.

In [ ]:
test_pred = best_model.predict(X_test[prediction_features])
test_metrics = regression_metrics(y_test, test_pred)

display(pd.DataFrame([test_metrics], index=[best_model_name]))

# 46. Create Error Table

**What this cell does:** Builds an error analysis table.

**Why we do this:** Metrics alone do not show individual prediction mistakes.

**How it works:** Residual is actual minus predicted, and absolute error is the size of the mistake.

In [ ]:
error_df = pd.DataFrame({
    "actual_fare": y_test,
    "predicted_fare": test_pred,
})
error_df["residual"] = error_df["actual_fare"] - error_df["predicted_fare"]
error_df["absolute_error"] = error_df["residual"].abs()

display(error_df.describe())

# 47. Plot Actual vs Predicted

**What this cell does:** Plots predicted fares against actual fares.

**Why we do this:** This visual shows whether predictions are close to perfect prediction.

**How it works:** Points near the red diagonal line are better predictions.

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=error_df["actual_fare"], y=error_df["predicted_fare"], alpha=0.25)
plt.plot(
    [error_df["actual_fare"].min(), error_df["actual_fare"].max()],
    [error_df["actual_fare"].min(), error_df["actual_fare"].max()],
    color="red",
)
plt.title("Actual vs Predicted Fare")
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.show()

# 48. Plot Residuals

**What this cell does:** Plots the distribution of residual errors.

**Why we do this:** Residuals should be centered near zero if the model is not strongly biased.

**How it works:** A histogram and KDE curve show the spread and shape of errors.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(error_df["residual"], bins=80, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

# 49. Extract Feature Importance

**What this cell does:** Extracts feature importance if the final model supports it.

**Why we do this:** Feature importance helps explain what drives fare prediction.

**How it works:** Tree models expose `feature_importances_`; linear models may skip this section.

In [ ]:
model_step = best_model.named_steps["model"]
preprocessor_step = best_model.named_steps["preprocessor"]

if hasattr(model_step, "feature_importances_"):
    feature_names = preprocessor_step.get_feature_names_out()
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model_step.feature_importances_,
    }).sort_values("importance", ascending=False)
    display(importance_df.head(25))
else:
    importance_df = None
    print(f"{best_model_name} does not expose feature_importances_.")

# 50. Plot Feature Importance

**What this cell does:** Plots top feature importance values when available.

**Why we do this:** A plot is easier to understand than a long table.

**How it works:** The top 20 rows from `importance_df` are shown as a horizontal bar chart.

In [ ]:
if importance_df is not None:
    plt.figure(figsize=(10, 8))
    sns.barplot(data=importance_df.head(20), x="importance", y="feature")
    plt.title(f"Top Feature Importances - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()
else:
    print("Feature importance plot is skipped for this model type.")

# 51. Save Final Model

**What this cell does:** Saves the selected model pipeline and metadata.

**Why we do this:** Saving allows the model to be reused in backend or production code without retraining.

**How it works:** `joblib.dump()` writes the pipeline, selected features, RFE results, and test metrics to disk.

In [ ]:
joblib.dump({
    "model_name": best_model_name,
    "pipeline": best_model,
    "prediction_features": prediction_features,
    "numeric_features": selected_numeric_features,
    "categorical_features": selected_categorical_features,
    "rfe_results": rfe_results,
    "test_metrics": test_metrics,
}, FARE_MODEL_PATH)

print("Saved fare prediction pipeline to:", FARE_MODEL_PATH)

# 52. Define Single-Ride Input Function

**What this cell does:** Creates one row of input features for a future ride.

**Why we do this:** The backend needs a repeatable way to create the exact feature format expected by the model.

**How it works:** The function accepts ride inputs and recreates the same time and route features used during training.

In [ ]:
def build_fare_input(
    pickup_datetime, passenger_count, trip_distance, ratecode_id,
    pu_location_id, do_location_id, predicted_trip_duration, cbd_congestion_fee=0.0,
):
    pickup_datetime = pd.to_datetime(pickup_datetime)

    return pd.DataFrame([{
        "passenger_count": passenger_count,
        "trip_distance": trip_distance,
        "RatecodeID": ratecode_id,
        "PULocationID": pu_location_id,
        "DOLocationID": do_location_id,
        "trip_duration": predicted_trip_duration,
        "cbd_congestion_fee": cbd_congestion_fee,
        "pickup_hour": pickup_datetime.hour,
        "pickup_dayofweek": pickup_datetime.dayofweek,
        "pickup_day": pickup_datetime.day,
        "is_weekend": int(pickup_datetime.dayofweek in [5, 6]),
        "is_rush_hour": int(pickup_datetime.hour in [7, 8, 9, 16, 17, 18, 19]),
        "Location_Pair": f"{pu_location_id}_{do_location_id}",
    }])

# 53. Test Single-Ride Prediction

**What this cell does:** Runs one example prediction with the final model.

**Why we do this:** This confirms the final input function and model pipeline work together.

**How it works:** A sample ride is converted to features, filtered to selected prediction columns, and passed to the model.

In [ ]:
sample_ride = build_fare_input(
    pickup_datetime="2026-01-15 08:30:00",
    passenger_count=1,
    trip_distance=3.2,
    ratecode_id=1,
    pu_location_id=239,
    do_location_id=161,
    predicted_trip_duration=18.5,
    cbd_congestion_fee=0.75,
)

predicted_fare = best_model.predict(sample_ride[prediction_features])[0]
print(f"Predicted Fare Amount: ${predicted_fare:.2f}")
display(sample_ride)

# Notebook Conclusion

1. The fare prediction dataset was prepared from cleaned taxi trip data.
2. Time, route, distance, passenger, rate-code, congestion, and duration features were created.
3. Initial models were trained and checked for overfitting using train-validation gaps.
4. Regularized models were used to reduce overfitting risk.
5. RFE was performed to identify stronger features.
6. Final models were retrained using RFE-selected features.
7. The best final model was evaluated on the independent test set.
8. The final pipeline was saved as `fare_prediction_pipeline.pkl`.
9. In deployment, `trip_duration` should come from your trip-duration prediction model.